# Public Key Infrastructure

In [1]:
import oqs
import json
import os
import uuid
import base64
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, UTC, timezone
import secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from enum import Enum, IntEnum
from pprint import pprint

liboqs-python faulthandler is disabled


### Phase 1 (Current)
Small Company PKI

Imagine a small software company called QuantumSoft Pvt Ltd.

It has

                QuantumSoft Root CA
                           │
             ┌─────────────┴─────────────┐
             │                           │
      Employee Intermediate CA     Server Intermediate CA
             │                           │
      ┌──────┴──────┐              ┌─────┴─────┐
      │             │              │           │
    Alice          Bob         Web Server   Mail Server

We'll create

Root Certificate Authority
Intermediate Certificate Authority
Employee Certificates
Server Certificates

using:

->Dilithium5 (Primary)
->Falcon-1024 (Experimental)
->Dilithium Backup wherever Falcon is used

No networking yet.



### Real World Story

Let's say we own a company called

QuantumSoft Pvt Ltd

The company has

Employees
  -Alice (Developer)
  -Bob (HR)
  -Charlie (Finance)

Each employee receives a certificate from Employee CA.

Servers
> mail.quantumsoft.com
> vpn.quantumsoft.com
>api.quantumsoft.com

Each server receives a certificate from Server CA.

Clients

Whenever Alice connects to

https://api.quantumsoft.com

her computer verifies

Server Certificate

↓

Server CA

↓

Root CA

↓

Trusted

Exactly how HTTPS works today.

##### Defining the organization

In [2]:
# ==========================================================
# SECTION 1: DEFINING THE ORGANIZATION
# ==========================================================

ORGANIZATION = {
    "Organization_name" : "QuantumSoft Pvt. Ltd.",
    "Country" : "IN",
    "state" : "Delhi",
    "city" : "New Delhi",
    "email": "security@quantumsoft.com"
}

print("Organization Initialized")
print(json.dumps(ORGANIZATION, indent=4))

Organization Initialized
{
    "Organization_name": "QuantumSoft Pvt. Ltd.",
    "Country": "IN",
    "state": "Delhi",
    "city": "New Delhi",
    "email": "security@quantumsoft.com"
}


##### Defining the cryptographic policy

In [3]:
# ==========================================================
# SECTION 2 : DEFINING THE CRYPTOGRAPHIC POLICY
# ==========================================================

CRYPTO_POLICY = {
    "primary_signature_algorithm": "ML-DSA-87",
    "experimental_signature_algorithm": "Falcon-1024",
    "backup_signature_algorithm": "ML-DSA-87",
    "kem_algorithm": "ML-KEM-1024",
    "hash_algorithm": "SHA-512"
}

print("Cryptographic Policy")
print(json.dumps(CRYPTO_POLICY, indent=4))

Cryptographic Policy
{
    "primary_signature_algorithm": "ML-DSA-87",
    "experimental_signature_algorithm": "Falcon-1024",
    "backup_signature_algorithm": "ML-DSA-87",
    "kem_algorithm": "ML-KEM-1024",
    "hash_algorithm": "SHA-512"
}


In [4]:
# ==========================================================
# SECTION 3 : DER ENCODER
# ==========================================================

class DEREncoder:

    TAGS = {
        "BOOLEAN": 0x01,
        "INTEGER": 0x02,
        "BIT_STRING": 0x03,
        "OCTET_STRING": 0x04,
        "NULL": 0x05,
        "OBJECT_IDENTIFIER": 0x06,
        "ENUMERATED": 0x0A,
        "UTF8_STRING": 0x0C,
        "SEQUENCE": 0x30,
        "SET": 0x31,
        "PRINTABLE_STRING": 0x13,
        "UTC_TIME": 0x17,
        "GENERALIZED_TIME": 0x18
    }

    @staticmethod
    def encode_length(length):

        if length<128:
            return bytes([length])
    
        length_bytes = []

        while length > 0:
            length_bytes.insert(0, length & 0xFF)
            length >>= 8

        return bytes([0x80 | len(length_bytes)]) + bytes(length_bytes)


    @staticmethod
    def encode_integer(value):

        if value < 0:
            raise ValueError("Negative Integers not Supported")

        integer_bytes = value.to_bytes(
            (value.bit_length() + 7)//8 or 1,
            "big"
        )

        #Prevent the value from being negative
        if integer_bytes[0] & 0x80:
            integer_bytes = b"\x00" + integer_bytes

        return (
            bytes([DEREncoder.TAGS["INTEGER"]]) +
            DEREncoder.encode_length(len(integer_bytes)) +
            integer_bytes
        )

    @staticmethod
    def encode_utf8_string(text: str) -> bytes:
        value = text.encode("utf-8")

        return(
            bytes([DEREncoder.TAGS["UTF8_STRING"]])+
            DEREncoder.encode_length(len(value))+
            value
        )

    @staticmethod
    def encode_printable_string(text: str) -> bytes:
        value = text.encode("ascii")

        return (
            bytes([DEREncoder.TAGS["PRINTABLE_STRING"]]) +
            DEREncoder.encode_length(len(value)) +
            value
        )

    @staticmethod
    def encode_iod_component(value: int) -> bytes:

        if value == 0:
            return b"\x00"

        result = []

        while value>0:
            result.insert(0, value & 0x7F)
            value >>= 7

        for i in range(len(result) - 1):
            result[i] |= 0x80

        return bytes(result)

    @staticmethod
    def encode_oid(oid: str) -> bytes:
        parts = [int(part) for part in oid.split('.')]

        if len(parts)<2:
            raise ValueError("OID must contain atleast 2 components")

        if parts[0]>2:
            raise ValueError("First OID component must be 0,1,2")


        combined = (40 * parts[0]) + parts[1]

        value = DEREncoder.encode_iod_component(combined)

        for component in parts[2:]:
            value += DEREncoder.encode_iod_component(component)

        return (
            bytes([DEREncoder.TAGS["OBJECT_IDENTIFIER"]]) +
            DEREncoder.encode_length(len(value)) +
            value
        )

    @staticmethod
    def encode_sequence(*elements: bytes) -> bytes:
        value = b"".join(elements)

        return (
            bytes([DEREncoder.TAGS["SEQUENCE"]]) +
            DEREncoder.encode_length(len(value)) +
            value
        )

    @staticmethod
    def encode_set(*elements: bytes) -> bytes:
        value = b"".join(elements)

        return (
            bytes([DEREncoder.TAGS["SET"]])+
            DEREncoder.encode_length(len(value))+
            value
        )

    @staticmethod
    def encode_bit_string(data: bytes, unused_bits: int = 0) -> bytes:


        if not (0 <= unused_bits <=7):
            raise ValueError("Unused bits must be between 0 and 7.")

        value = bytes([unused_bits])+data

        return (
            bytes([DEREncoder.TAGS["BIT_STRING"]]) +
            DEREncoder.encode_length(len(value)) +
            value
        )

    @staticmethod
    def encode_null() -> bytes:

        return bytes([
            DEREncoder.TAGS["NULL"],
            0x00
        ])

    @staticmethod
    def encode_utctime(dt: datetime) -> bytes:
        value = dt.strftime("%y%m%d%H%M%SZ").encode("ascii")

        return(
            bytes([DEREncoder.TAGS["UTC_TIME"]])+
            DEREncoder.encode_length(len(value))+
            value
        )

    @staticmethod
    def encode_generalized_time(dt: datetime) -> bytes:

        value = dt.strftime("%Y%m%d%H%M%SZ").encode("ascii")

        return(
            bytes([DEREncoder.TAGS["GENERALIZED_TIME"]])+
            DEREncoder.encode_length(len(value))+
            value
        )

    @staticmethod
    def encode_time(dt: datetime) -> bytes:

        if dt.year<=2049:
            return DEREncoder.encode_utctime(dt)

        return DEREncoder.encode_generalized_time(dt)


    @staticmethod
    def encode_explicit_context(tag_number: int, content: bytes) -> bytes:

        if not(0<=tag_number<=30):
            raise ValueError("Only Tag Numbers 0 to 30 supported!!!")

        tag = DEREncoder._context_tag(tag_number, True)

        return(
            bytes([tag])+
            DEREncoder.encode_length(len(content))+
            content
        )

    @staticmethod
    def encode_octet_string(data: bytes) -> bytes:
        return (
            bytes([DEREncoder.TAGS['OCTET_STRING']])+
            DEREncoder.encode_length(len(data))+
            data
        )

    @staticmethod 
    def encode_boolean(value: bool) -> bytes:

        encoded_value = b"\xFF" if value else b"\x00"

        return (
            bytes([DEREncoder.TAGS["BOOLEAN"]])+
            DEREncoder.encode_length(1)+
            encoded_value
        )

    @staticmethod
    def encode_implicit_context(tag_number: int, encoded_values: bytes) -> bytes:

        if not encoded_values:
            raise ValueError("Encoded Value cannot be empty!!!!")

        context_tag = DEREncoder._context_tag(tag_number, False)

        return (bytes([context_tag]) + encoded_values[1:])

    @staticmethod
    def _context_tag(tag_number: int, constructed: bool) -> int:
        if not (0 <= tag_number <= 30):
            raise ValueError("Context-specific tag number must be between 0 and 30.")

        base = 0x80

        if constructed:
            base |= 0x20

        return base | tag_number


    @staticmethod
    def encode_enumerated(value: int) -> bytes:

        if value<0:
            raise ValueError("Negative ENUMERATED values are not supported!!!")

        encoded = value.to_bytes((value.bit_length() + 7)//8 or 1, 'big')

        if encoded[0] & 0x80:
            encoded = b"\x00" + encoded

        return (
            bytes([DEREncoder.TAGS["ENUMERATED"]])+
            DEREncoder.encode_length(len(encoded)) +
            encoded
        )





    

In [5]:
# ==========================================================
# SECTION 4 : UTILITY FUNCTIONS
# ==========================================================



class RevocationReason(IntEnum):
    """
    X.509 Certificate Revocation Reasons
    RFC 5280 - CRLReason
    """

    UNSPECIFIED = 0
    KEY_COMPROMISE = 1
    CA_COMPROMISE = 2
    AFFILIATION_CHANGED = 3
    SUPERSEDED = 4
    CESSATION_OF_OPERATION = 5
    CERTIFICATE_HOLD = 6
    REMOVE_FROM_CRL = 8
    PRIVILEGE_WITHDRAWN = 9
    AA_COMPROMISE = 10

    
def hex_dump(data):
    return " ".join(f"{byte:02X}" for byte in data)

class ParameterEncoding(Enum):

    ABSENT = 0
    
    NULL = 1
    
    CUSTOM = 2  


class KeyState:

    ACTIVE = "ACTIVE"

    ROTATED = "ROTATED"

    DESTROYED = "DESTROYED"

def generate_serial_number()->int:
    return secrets.randbits(159)


def inspect_entity(entity):

    print("=" * 70)

    print(entity["name"][-1][1])

    print("=" * 70)

    print("\nIdentity")
    print("--------")

    for attribute, value in entity["name"]:
        print(f"{attribute:30} : {value}")

    print("\nSignature")
    print("---------")

    print(f"Algorithm              : {entity['signature']['algorithm']}")
    print(f"OQS Algorithm          : {entity['signature']['oqs_algorithm']}")

    print(f"Public Key Generated   : {entity['signature']['public_key'] is not None}")
    print(f"Private Key Generated  : {entity['signature']['private_key'] is not None}")

    print("\nCertificate")
    print("-----------")

    if entity["certificate"] is None:

        print("No certificate issued.")

    else:

        print(f"Serial Number          : {entity['certificate']['serial_number']}")
        print(f"Issuer                 : {entity['certificate']['issuer']}")
        print(f"Status                 : {entity['certificate']['status']}")
        print(f"Issued At              : {entity['certificate']['issued_at']}")
        print(f"Expires At             : {entity['certificate']['expires_at']}")
        print(f"Certificate Size       : {len(entity['certificate']['certificate'])} bytes")

    print("=" * 70)

def inspect_ca_database():

    print("=" * 90)
    print("           QuantumSoft Root CA Certificate Database")
    print("=" * 90)

    print(f"Total Certificates : {len(ROOT_CA['issued_certificates'])}")

    print()

    for record in ROOT_CA["issued_certificates"]:

        subject = dict(record["subject"])

        print(f"Serial Number : {record['serial_number']}")
        print(f"Common Name   : {subject['COMMON_NAME']}")
        print(f"Status        : {record['status']}")
        print(f"Issued At     : {record['issued_at']}")
        print(f"Expires At    : {record['expires_at']}")
        print("-" * 90)


def inspect_certificate(entity):

    cert = entity["certificate"]

    print("=" * 60)
    print(f"Subject      : {cert['subject']}")
    print(f"Issuer       : {cert['issuer']}")
    print(f"Serial       : {cert['serial_number']}")
    print(f"Status       : {cert['status']}")
    print(f"Issued At    : {cert['issued_at']}")
    print(f"Expires At   : {cert['expires_at']}")
    print(f"Certificate Size : {len(cert['certificate'])} bytes")
    print("=" * 60)


def inspect_entity(entity):

    print("=" * 80)
    print(f"Entity Type      : {entity['type']}")
    print(f"Entity ID        : {entity['entity_id']}")
    print(f"Common Name      : {entity['common_name']}")
    print(f"Organization     : {entity['organization']}")

    print("\n----- Distinguished Name -----")
    for key, value in entity["name"].items():
        print(f"{key:15}: {value}")

    print("\n----- Parent CA -----")
    if entity["parent_ca"]:
        print(entity["parent_ca"]["common_name"])
    else:
        print("None (Self Signed Root)")

    print("\n----- Signature -----")
    print("Algorithm        :", entity["signature"]["algorithm"])
    print("Public Key Size  :", len(entity["signature"]["public_key"]), "bytes")
    print(
        "Private Key      :",
        "Available" if entity["signature"]["private_key"] else "Destroyed"
    )

    print("\n----- Certificate -----")
    certificate = entity["certificate"]

    print("Serial Number    :", certificate["serial_number"])
    print("Signature Algo   :", entity["signature"]["algorithm"])
    print("DER Size         :", len(certificate["certificate"]), "bytes")

    print("=" * 80)


In [6]:
# ==========================================================
# SECTION 5 : REGISTRIES
# ==========================================================

OID_REGISTRY = {

    "country": {
        "oid": "2.5.4.6",
        "string_type": "PRINTABLE_STRING"
    },

    "state": {
        "oid": "2.5.4.8",
        "string_type": "UTF8_STRING"
    },

    "locality": {
        "oid": "2.5.4.7",
        "string_type": "UTF8_STRING"
    },

    "organization": {
        "oid": "2.5.4.10",
        "string_type": "UTF8_STRING"
    },

    "common_name": {
        "oid": "2.5.4.3",
        "string_type": "UTF8_STRING"
    }

}

ALGORITHM_IDENTIFIER_REGISTRY = {

    "RSA": {
        "oid": "1.2.840.113549.1.1.1",
        "oqs_name": None,
        "famaly" : "RSA",
        "hash": "SHA-256",
        "parameter_encoding": ParameterEncoding.NULL
    },

    "ECDSA": {
        "oid": "1.2.840.10045.2.1",
        "oqs_name": None,
        "family": "ECDSA",
        "hash": "SHA-256",
        "parameter_encoding": ParameterEncoding.ABSENT
    },

    "ML_DSA_44": {
        "oid": "2.16.840.1.101.3.4.3.17",
        "oqs_name": "ML-DSA-44",
        "family": "ML-DSA",
        "hash": "SHA3-256",
        "parameter_encoding": ParameterEncoding.ABSENT
    },

    "ML_DSA_65": {
        "oid": "2.16.840.1.101.3.4.3.18",
        "oqs_name": "ML-DSA-65",
        "family": "ML_DSA",
        "hash": "SHA3-384",
        "parameter_encoding": ParameterEncoding.ABSENT
    },

    "ML_DSA_87": {
        "oid": "2.16.840.1.101.3.4.3.19",
        "oqs_name": "ML-DSA-87",
        "family": "ML_DSA",
        "hash": "SHA3-512",
        "parameter_encoding": ParameterEncoding.ABSENT
    },

    "FALCON_512": {
        "oid": None,
        "oqs_name": "Falcon-512",
        "family": "Falcon",
        "hash": "SHAKE256",
        "parameter_encoding": ParameterEncoding.ABSENT
    },

    "FALCON_1024": {
        "oid": None,
        "oqs_name": "Falcon-1024",
        "family": "Falcon",
        "hash": "SHAKE256",
        "parameter_encoding": ParameterEncoding.ABSENT
    }

}

EXTENSION_REGISTRY = {

    "BASIC_CONSTRAINTS": {

        "oid": "2.5.29.19",

        "critical": True

    },

    "KEY_USAGE": {

        "oid": "2.5.29.15",

        "critical": True

    },

    "SUBJECT_KEY_IDENTIFIER": {

        "oid": "2.5.29.14",

        "critical": False

    },

    "AUTHORITY_KEY_IDENTIFIER": {

        "oid": "2.5.29.35",

        "critical": False

    },

    "CRL_REASON": {
    
        "oid": "2.5.29.21",
    
        "critical": False
    },

    "CRL_NUMBER": {
        
        "oid": "2.5.29.20",
        
        "critical": False
        
    }

}


In [7]:
# ==========================================================
# SECTION 6 : X.509 BUILDERS
# ==========================================================

class X509Builder:

    @staticmethod
    def build_attribute(attribute_name, value):

        attribute = OID_REGISTRY[attribute_name]

        encoded_oid = DEREncoder.encode_oid(
            attribute["oid"]
        )

        if attribute["string_type"] == "PRINTABLE_STRING":
            encoded_value = DEREncoder.encode_printable_string(value)

        else:
            encoded_value = DEREncoder.encode_utf8_string(value)


        return DEREncoder.encode_sequence(
            encoded_oid,
            encoded_value
        )

    @staticmethod
    def build_name(attributes: dict) -> bytes:


        order = [
            "country",
            "state",
            "locality",
            "organization",
            "common_name"
        ]

        rdn_sequence = []

        for attribute_name in order:

            if attribute_name not in attributes:

                continue

            attribute = X509Builder.build_attribute(
                attribute_name,
                attributes[attribute_name]
            )

            rdn_sequence.append(DEREncoder.encode_set(attribute))

            

        return DEREncoder.encode_sequence(*rdn_sequence)

    @staticmethod
    def build_algorithm_identifier(
        algorithm_name: str,
        custom_parameters: bytes | None = None
    ) -> bytes:

        algorithm = ALGORITHM_IDENTIFIER_REGISTRY[algorithm_name]

        if algorithm["oid"] is None:
            raise ValueError(
                f"No official OID available for {algorithm_name}"
            )

        elements = [
            DEREncoder.encode_oid(
                algorithm["oid"]
            )
        ]

        match algorithm["parameter_encoding"]:

            case ParameterEncoding.NULL:
                elements.append(DEREncoder.encode_null())

            case ParameterEncoding.CUSTOM:
                if custom_parameters is None:
                    raise ValueError("Custom Parameters Required")
                elements.append(custom_parameters)

            case ParameterEncoding.ABSENT:
                pass

        return DEREncoder.encode_sequence(*elements)   


    @staticmethod
    def build_subject_public_key_info(algorithm_name: str, public_key: bytes) -> bytes:
        
        algorithm_identifier = (
            X509Builder.build_algorithm_identifier(
                algorithm_name
            )
        )

        encoded_public_key = (
            DEREncoder.encode_bit_string(
                public_key
            )
        )

        return DEREncoder.encode_sequence(
            algorithm_identifier,
            encoded_public_key
        )

    @staticmethod
    def build_validity(
        not_before: datetime,
        not_after: datetime
    ) -> bytes:

        return DEREncoder.encode_sequence(

            DEREncoder.encode_time(not_before),

            DEREncoder.encode_time(not_after)
        )

    @staticmethod
    def build_version(version: int = 3) -> bytes:

        if version not in (1,2,3):
            raise ValueError("Only X.509 versions 1, 2, 3 are supported!!!")

        encoded_version = DEREncoder.encode_integer(version-1)

        return DEREncoder.encode_explicit_context(
            0,
            encoded_version
        )

    @staticmethod
    def build_crl_version(version: int) -> bytes:

        if version not in (1,2):
            raise ValueError("CRL version must be 1 or 2.")

        return DEREncoder.encode_integer(version - 1)

    @staticmethod
    def build_serial_number(serial: int) -> bytes:

        return DEREncoder.encode_integer(serial)

    @staticmethod
    def build_tbs_certificate(
        version: bytes,
        serial_number: bytes,
        signature_algorithm: str,
        issuer: bytes,
        validity: bytes,
        subject: bytes,
        subject_public_key_info: bytes,
        extensions
        
        
    )-> bytes:


        signature = X509Builder.build_algorithm_identifier(signature_algorithm)

        return DEREncoder.encode_sequence(
            version,
            serial_number,
            signature,
            issuer,
            validity,
            subject,
            subject_public_key_info,
            extensions
        )

    @staticmethod
    def build_certificate(tbs_certificate: bytes, signature_algorithm: str, signature: bytes) -> bytes:
        
        algorithm_identifier = (X509Builder.build_algorithm_identifier(signature_algorithm))

        signature_value = DEREncoder.encode_bit_string(signature)

        return DEREncoder.encode_sequence(
            tbs_certificate,
            algorithm_identifier,
            signature_value
        )

    @staticmethod
    def build_extension(extension_name: str, extension_value: bytes) -> bytes:
        
        extension = EXTENSION_REGISTRY[extension_name]

        elements = [DEREncoder.encode_oid(extension['oid'])]

        if extension['critical']:
            elements.append(DEREncoder.encode_boolean(True))


        elements.append(DEREncoder.encode_octet_string(extension_value))

        return DEREncoder.encode_sequence(*elements)

    @staticmethod
    def build_extensions(*extensions: bytes) -> bytes:

        if len(extensions) == 0:
            return b""

        extensions_sequence = DEREncoder.encode_sequence(*extensions)

        return DEREncoder.encode_explicit_context(3,extensions_sequence)


    @staticmethod
    def build_crl_extensions_container(*extensions: bytes) -> bytes:

        if len(extensions) == 0:
            return b""

        extensions_sequence = DEREncoder.encode_sequence(*extensions)

        return DEREncoder.encode_explicit_context(0,extensions_sequence)

    @staticmethod
    def build_basic_constraints(is_ca: bool, path_length: int | None = None) -> bytes:

        elements = []

        elements.append(DEREncoder.encode_boolean(is_ca))

        if path_length is not None:
            elements.append(DEREncoder.encode_integer(path_length))

        return DEREncoder.encode_sequence(*elements)

    @staticmethod
    def build_key_usage(
        digital_signature: bool = False,
        non_repudiation: bool = False,
        key_encipherment: bool = False,
        data_encipherment: bool = False,
        key_agreement: bool = False,
        key_cert_sign: bool = False,
        crl_sign: bool = False,
        encipher_only: bool = False,
        decipher_only: bool = False
    ) -> bytes:

        flags = [
            digital_signature,
            non_repudiation,
            key_encipherment,
            data_encipherment,
            key_agreement,
            key_cert_sign,
            crl_sign,
            encipher_only,
            decipher_only
        ]

        last_set = -1

        for index, enabled in enumerate(flags):

            if enabled:
                last_set = index

        unused_bits = (8-(len(flags)%8)) %8

        flags.extend([False] * unused_bits)

        encoded = bytearray()

        for i in range(0,len(flags), 8):

            value = 0

            for bit in flags[i:i+8]:

                value <<=1

                if bit:
                    value|=1

            encoded.append(value)

        return DEREncoder.encode_bit_string(bytes(encoded), unused_bits)


    @staticmethod
    def build_subject_key_identifier(public_key: bytes) -> bytes:

        digest = hashlib.sha1(public_key).digest()

        return DEREncoder.encode_octet_string(digest)


    @staticmethod
    def build_authority_key_identifier(issuer_public_key: bytes) -> bytes:

        digest = hashlib.sha1(issuer_public_key).digest()

        identifier = DEREncoder.encode_octet_string(digest)

        identifier = DEREncoder.encode_implicit_context(0,identifier)

        return DEREncoder.encode_sequence(identifier)

    @staticmethod
    def build_default_extensions(
        issuer: dict,
        subject: dict
    ) -> bytes:

        extensions = []

        extensions_data = {}

        match subject["type"]:
    
            case "ROOT_CA":

                basic_constraints = {
                    "ca" : True,
                    "path_length" : 1
                }

                extensions_data["basic_constraints"] = basic_constraints
    
                extensions.append(
                    X509Builder.build_extension(
                        "BASIC_CONSTRAINTS",
                        X509Builder.build_basic_constraints(
                            is_ca=True,
                            path_length=1
                        )
                    )
                )

                key_usage = {
                    "digital_signature": False,
                    "non_repudiation": False,
                    "key_encipherment": False,
                    "data_encipherment": False,
                    "key_agreement": False,
                    "key_cert_sign": True,
                    "crl_sign": True,
                    "encipher_only": False,
                    "decipher_only": False
                }

                extensions_data["key_usage"] = key_usage
    
                extensions.append(
                    X509Builder.build_extension(
                        "KEY_USAGE",
                        X509Builder.build_key_usage(
                            key_cert_sign=True,
                            crl_sign=True
                        )
                    )
                )
    
            case "INTERMEDIATE_CA":

                basic_constraints = {
                    "ca" : True,
                    "path_length" : 0
                }

                extensions_data["basic_constraints"] = basic_constraints
    
                extensions.append(
                    X509Builder.build_extension(
                        "BASIC_CONSTRAINTS",
                        X509Builder.build_basic_constraints(
                            is_ca=True,
                            path_length=0
                        )
                    )
                )

                key_usage = {
                    "digital_signature": False,
                    "non_repudiation": False,
                    "key_encipherment": False,
                    "data_encipherment": False,
                    "key_agreement": False,
                    "key_cert_sign": True,
                    "crl_sign": True,
                    "encipher_only": False,
                    "decipher_only": False
                }

                extensions_data["key_usage"] = key_usage
    
                extensions.append(
                    X509Builder.build_extension(
                        "KEY_USAGE",
                        X509Builder.build_key_usage(
                            key_cert_sign=True,
                            crl_sign=True
                        )
                    )
                )
    
            case "EMPLOYEE":


                basic_constraints = {
                    "ca" : False,
                }

                extensions_data["basic_constraints"] = basic_constraints
    
                extensions.append(
                    X509Builder.build_extension(
                        "BASIC_CONSTRAINTS",
                        X509Builder.build_basic_constraints(
                            is_ca=False
                        )
                    )
                )


                key_usage = {
                    "digital_signature": True,
                    "non_repudiation": False,
                    "key_encipherment": False,
                    "data_encipherment": False,
                    "key_agreement": False,
                    "key_cert_sign": False,
                    "crl_sign": False,
                    "encipher_only": False,
                    "decipher_only": False
                }

                extensions_data["key_usage"] = key_usage
    
                extensions.append(
                    X509Builder.build_extension(
                        "KEY_USAGE",
                        X509Builder.build_key_usage(
                            digital_signature=True
                        )
                    )
                )
    
            case "SERVER":

                basic_constraints = {
                    "ca" : False
                }

                extensions_data["basic_constraints"] = basic_constraints
    
                extensions.append(
                    X509Builder.build_extension(
                        "BASIC_CONSTRAINTS",
                        X509Builder.build_basic_constraints(
                            is_ca=False
                        )
                    )
                )

                key_usage = {
                    "digital_signature": True,
                    "non_repudiation": False,
                    "key_encipherment": False,
                    "data_encipherment": False,
                    "key_agreement": False,
                    "key_cert_sign": False,
                    "crl_sign": False,
                    "encipher_only": False,
                    "decipher_only": False
                }

                extensions_data["key_usage"] = key_usage
    
                extensions.append(
                    X509Builder.build_extension(
                        "KEY_USAGE",
                        X509Builder.build_key_usage(
                            digital_signature=True,
                            key_encipherment=True
                        )
                    )
                )
    
            case _:
    
                raise ValueError(
                    f"Unsupported certificate type: {subject['type']}"
                )

        ski = hashlib.sha1(subject["signature"]["public_key"]).digest()

        extensions_data["subject_key_identifier"] = ski

        extensions.append(
            X509Builder.build_extension(
                "SUBJECT_KEY_IDENTIFIER",
                X509Builder.build_subject_key_identifier(
                    subject["signature"]["public_key"]
                )
            )
        )

        aki = hashlib.sha1(issuer["signature"]["public_key"]).digest()

        extensions_data["authority_key_identifier"] = aki
    
        extensions.append(
            X509Builder.build_extension(
                "AUTHORITY_KEY_IDENTIFIER",
                X509Builder.build_authority_key_identifier(
                    issuer["signature"]["public_key"]
                )
            )
        )
    
        return X509Builder.build_extensions(*extensions), extensions_data




    @staticmethod
    def build_reason_code(reason:RevocationReason) -> bytes:

        return DEREncoder.encode_enumerated(reason.value)


    @staticmethod
    def build_revoked_certificate_extensions(reason: RevocationReason, invalidity_date: datetime | None = None) -> tuple[bytes, dict]:

        extensions = []

        extensions_data = {}

        extensions_data["reason"] = reason.name

        extensions.append(
            X509Builder.build_extension(
                "CRL_REASON",
                X509Builder.build_reason_code(reason)
            )
        )


        return DEREncoder.encode_sequence(*extensions), extensions_data



    @staticmethod
    def build_revoked_certificate(serial_number: int, revocation_date: datetime, reason: RevocationReason) -> tuple[bytes, dict]:


        extension, extension_data = X509Builder.build_revoked_certificate_extensions(reason)


        revoked_certificate = DEREncoder.encode_sequence(
            DEREncoder.encode_integer(serial_number),
            DEREncoder.encode_time(revocation_date),
            extension
        )


        revoked_certificate_data = {
            "serial_number": serial_number,
            "revocation_date": revocation_date,
            "extensions": extension_data
        }



        return revoked_certificate, revoked_certificate_data



    

    @staticmethod
    def build_revoked_certificates(revoked_records: list) -> tuple[bytes, list]:

        revoked_certificates_list = []
        revoked_certificates_data_list = []

        for record in revoked_records:

            revoked_certificates, revoked_certificates_data = (
                X509Builder.build_revoked_certificate(
                    serial_number=record["serial_number"],
                    revocation_date=record["revocation_date"],
                    reason=record["revocation_reason"]
                )
            )

            revoked_certificates_list.append(revoked_certificates)
            revoked_certificates_data_list.append(revoked_certificates_data)


        return DEREncoder.encode_sequence(*revoked_certificates_list), revoked_certificates_data_list




    @staticmethod
    def build_tbs_crl(version: bytes, signature_algorithm: str, issuer: bytes, this_update: datetime, next_update: datetime, revoked_certificates: bytes, extensions: bytes) -> bytes:

        signature = X509Builder.build_algorithm_identifier(signature_algorithm)


        return DEREncoder.encode_sequence(

            version,
            signature,
            issuer,
            DEREncoder.encode_time(this_update),
            DEREncoder.encode_time(next_update),
            revoked_certificates,
            extensions
        )






    @staticmethod
    def build_crl_number(num: int) -> bytes:

        return DEREncoder.encode_integer(num)




        
    @staticmethod
    def build_crl_extensions(issuer: dict) -> tuple[bytes, dict]:

        extensions = []

        extensions_data = {}


        extensions_data["authority_key_identifier"] = hashlib.sha1(issuer["signature"]["public_key"]).digest()
        extensions.append(X509Builder.build_extension("AUTHORITY_KEY_IDENTIFIER", X509Builder.build_authority_key_identifier(issuer["signature"]["public_key"])))


        extensions_data["crl_number"] = issuer["next_crl_number"]
        extensions.append(X509Builder.build_extension("CRL_NUMBER", X509Builder.build_crl_number(issuer["next_crl_number"])))


        

        return X509Builder.build_crl_extensions_container(*extensions), extensions_data
        


        
    @staticmethod
    def build_crl(tbs_crl: bytes, signature_algorithm: str, signature: bytes) -> bytes:

        algorithm_identifier = (X509Builder.build_algorithm_identifier(signature_algorithm))

        signature_value = DEREncoder.encode_bit_string(signature)

        return DEREncoder.encode_sequence(tbs_crl, algorithm_identifier, signature_value)



    

We have created the structure of X.509 certificate and we are ready to build a certificate for the users

Now we created the Root CA and its certificate and also verified it..... Now we will proceed with creating a first user's CA.

##### The procedure of issueing Alice CA is:

Alice joins QuantumSoft Pvt. Ltd.

        │
        ▼

Alice generates her own ML-DSA key pair

        │
        ▼

Private Key  ← stays with Alice forever

Public Key

        │
        ▼

Alice sends a Certificate Signing Request (CSR)

        │
        ▼

QuantumSoft Root CA verifies Alice's identity

        │
        ▼

Root CA signs Alice's public key

        │
        ▼

Alice receives her certificate

##### What is a CSR?

A CSR (Certificate Signing Request) is simply a request saying:

Hello QuantumSoft Root CA,

My name is Alice.

Here is my public key.

Please issue a certificate for me.

Regards,
Alice

##### Cryptographically, a CSR contains:

CSR
│
├── Subject
│      CN = Alice Sharma
│
├── Public Key
│
├── Requested Extensions
│
└── Signature
      (Created using Alice's private key)

In [8]:
# ==========================================================
# SECTION 7 : ENTITY FACTORY
# ==========================================================


class EntityFactory:

    @staticmethod
    def create_root_ca(country: str,state: str,locality: str,  organization: str, common_name: str) -> dict:

        return EntityFactory._create_entity(

            entity_type = "ROOT_CA",
            organization = organization,
            common_name = common_name,
            country = country,
            locality = locality,
            state = state,
            issued_certificates = [],
            crls = [],
            next_crl_number = 1,
            statistics = {
                "issued_certificates": 0,
                "revoked_certificates": 0,
                "generated_crls": 0
            },
            next_serial_number = 1
        )

        



    @staticmethod
    def create_intermediate_ca(country: str, state: str, locality:str, organization: str, common_name: str) -> dict:

        return EntityFactory._create_entity(

            entity_type = "INTERMEDIATE_CA",
            country = country,
            state = state,
            locality = locality,
            organization = organization,
            common_name = common_name,
            issued_certificates = [],
            crls = [],
            next_crl_number = 1,
            statistics = {
                "issued_certificates": 0,
                "revoked_certificates": 0,
                "generated_crls": 0
            },
            next_serial_number = 1
        )




    @staticmethod
    def create_employee(country: str, state: str, locality:str, organization: str, common_name: str, employee_id: str) -> dict:


        return EntityFactory._create_entity(

            entity_type = "EMPLOYEE",
            country = country,
            state = state,
            locality = locality,
            organization = organization,
            common_name = common_name,
            employee_id = employee_id
        )




    @staticmethod
    def create_server(country: str, state: str, locality:str, organization: str, common_name: str, dns_name: str) -> dict:

        return EntityFactory._create_entity(

            entity_type = "SERVER",
            country = country,
            state = state,
            locality = locality,
            organization = organization,
            common_name = common_name,
            dns_name = dns_name
        )

    @staticmethod
    def _create_entity(entity_type: str, country: str, state: str, locality:str, organization: str, common_name: str, **extra_fields) -> dict:

        entity = {

            #Entity Information

            "entity_id": str(uuid.uuid4()),

            "type": entity_type,
            "organization": organization,
            "common_name": common_name,

            #DER encoded X.509
            "name":{
                
                "common_name" : common_name,
                "organization": organization,
                "country": country,
                "state": state,
                "locality": locality
                
            },

            #Cryptograohic Material

            "signature": None,
            "certificate": None,

            #Creation Time

            "created_at" : datetime.now(timezone.utc),

            #PKI Relationship

            "parent_ca": None,

        }

        entity.update(extra_fields)

        return entity

    @staticmethod
    def display(entity):
    
        print("=" * 80)
        print(f"{entity['type']} INFORMATION")
        print("=" * 80)
    
        print(f"Entity ID      : {entity['entity_id']}")
        print(f"Common Name    : {entity['common_name']}")
        print(f"Organization   : {entity['organization']}")
        print(f"Created At     : {entity['created_at']}")
    
        print("\nDistinguished Name")
        print("------------------")
    
        for k, v in entity["name"].items():
            print(f"{k:<15}: {v}")
    
        print()
    
        if entity["type"] == "EMPLOYEE":
            print(f"Employee ID    : {entity['employee_id']}")
    
        if entity["type"] == "SERVER":
            print(f"DNS Name       : {entity['dns_name']}")
    
        print()
    
        print(f"Signature      : {'Initialized' if entity['signature'] else 'Not Initialized'}")
        print(f"Certificate    : {'Issued' if entity['certificate'] else 'Not Issued'}")
    
        if entity["parent_ca"]:
            print(f"Parent CA      : {entity['parent_ca']['common_name']}")
        else:
            print("Parent CA      : None")
    
        if entity["type"] in ("ROOT_CA", "INTERMEDIATE_CA"):
    
            print("\nCA INFORMATION")
            print("------------------")
    
            print(f"Issued Certs   : {len(entity['issued_certificates'])}")
            print(f"CRLs           : {len(entity['crls'])}")
            print(f"Next Serial    : {entity['next_serial_number']}")
    
            print("\nStatistics")
            print("----------")
    
            for k, v in entity["statistics"].items():
                print(f"{k:<25}: {v}")
    
        print("=" * 80)

    

In [9]:
# ==========================================================
# SECTION 8 : CRYPTOGRAPHIC FUNCTIONS
# ==========================================================


class CryptoManager:


    DEFAULT_SIGNATURE_ALGORITHM = "ML_DSA_87"

    DEFAULT_HASH = "SHA-512"



    # =====================================================
    # Signature Identity Management
    # =====================================================

    
    @staticmethod
    def initialize_signature(entity: dict, algorithm: str = DEFAULT_SIGNATURE_ALGORITHM) -> None:

        algorithm_info = ALGORITHM_IDENTIFIER_REGISTRY[algorithm]

        with oqs.Signature(algorithm_info["oqs_name"]) as signer:

            public_key = signer.generate_keypair()

            private_key = signer.export_secret_key()


        entity["signature"] = {

            "algorithm": algorithm,

            "family": algorithm_info["family"],

            "hash": algorithm_info["hash"],

            "oqs_name": algorithm_info["oqs_name"],

            "oid": algorithm_info["oid"],

            "public_key": public_key,

            "private_key": private_key,

            "key_state": KeyState.ACTIVE
        }

        

    @staticmethod
    def rotate_signature_keys(entity: dict, algorithm: str = DEFAULT_SIGNATURE_ALGORITHM) -> None:

        signature = entity["signature"]

        if signature is None:

            raise ValueError("Entity has no cryptographic identity!!!!")

        if algorithm is None:

            algorithm = entity["signature"]["algorithm"]

        signature["key_state"] = KeyState.ROTATED

        CryptoManager.destroy_private_key(entity)

        CryptoManager.initialize_signature(entity, algorithm)


    # =====================================================
    # Signature Operations
    # =====================================================


    @staticmethod
    def sign(entity: dict, data: bytes) -> bytes:

        
        if entity["signature"] is None:
            raise ValueError("Entity has no cryptographic identity!!!")

        signature_info = entity["signature"]

        if signature_info["key_state"] != KeyState.ACTIVE:
            raise ValueError("Private key is not active!!!")

        with oqs.Signature(signature_info["oqs_name"], signature_info["private_key"]) as signer:

            return signer.sign(data)
            


    @staticmethod
    def verify(entity: dict, data: bytes, signature: bytes) -> bool:

        signature_info = entity.get("signature")

        if signature_info is None:
            raise ValueError("Entity has no cryptographic identity!!!")

        if "public_key" not in signature_info:
            raise ValueError("Missing Public Key!!!")

        if "oqs_name" not in signature_info:
            raise ValueError("Missing Signature Algorithm!!!")
            

        

        with oqs.Signature(signature_info["oqs_name"]) as verifier:

            return verifier.verify(data, signature, signature_info["public_key"])



    # =====================================================
    # Key Export
    # =====================================================


    @staticmethod
    def export_public_key(entity: dict) -> bytes:

        return entity["signature"]["public_key"]


    @staticmethod
    def export_private_key(entity: dict) -> bytes:

        signature = entity["signature"]

        if signature["key_state"] == KeyState.DESTROYED:

            raise ValueError("Private Key has been destroyed!!!")

        return entity["signature"]["private_key"]


    # =====================================================
    # Key Destruction
    # =====================================================

    @staticmethod
    def destroy_private_key(
        entity: dict
    ) -> None:

        signature = entity["signature"]

        if signature is None:

            raise ValueError("Entity has no cryptographic identity!!!")

        signature["private_key"] = None

        signature["key_state"] = KeyState.DESTROYED



    # =====================================================
    # Generate Session Key
    # =====================================================

    

    @staticmethod
    def generate_session_key(length: int = 32) -> bytes:

        return secrets.token_bytes(length)


    @staticmethod
    def generate_nonce(length: int = 12) -> bytes:

        return secrets.token_bytes(length)



    
    # =====================================================
    # AES Encryption
    # =====================================================


    @staticmethod
    def encrypt(plaintext: bytes, session_key: bytes, nonce: bytes) -> bytes:

        aes = AESGCM(session_key)

        return aes.encrypt(nonce, plaintext, None)




    @staticmethod
    def decrypt(ciphertext: bytes, session_key: bytes, nonce: bytes) -> bytes:


        aes = AESGCM(session_key)

        return aes.decrypt(nonce, ciphertext, None)



    # =====================================================
    # Print
    # =====================================================


    @staticmethod
    def print_signature_info(entity: dict) -> None:
    
        signature = entity["signature"]
    
        if signature is None:
            print("No signature initialized.")
            return
    
        print("Algorithm :", signature["algorithm"])
        print("Family    :", signature["family"])
        print("Hash      :", signature["hash"])
        print("OID       :", signature["oid"])
        print("State     :", signature["key_state"])
        print("Public Key:", len(signature["public_key"]), "bytes")
    
        if signature["private_key"] is None:
            print("Private Key: DESTROYED")
        else:
            print("Private Key:", len(signature["private_key"]), "bytes")
    

        

In [10]:
class CertificateAuthority:

    # ==========================================================
    # SECTION 9 : CERTIFICATE ISSUING
    # ==========================================================

    
    @staticmethod
    def issue_certificate(issuer_entity: dict, subject_entity: dict, validity_days = 365) -> bytes:
    
        """
        Issue an X.509 certificate for a subject.
    
        Parameters
        ----------
        subject_entity : dict
            Entity requesting the certificate
            (e.g. Alice, Bob, VPN Server)
    
        Returns
        -------
        bytes
            DER encoded X.509 certificate.
        """


        # ---------------------------------------
        # Validate cryptographic identities
        # ---------------------------------------

        if issuer_entity["signature"] is None:
            raise ValueError("Issuer has no cryptographic identity.")

        if subject_entity["signature"] is None:
            raise ValueError("Subject has no cryptographic identity.")

        
    
        # ---------------------------------------
        # Allocate Serial Number
        # ---------------------------------------
    
    
        serial = CertificateAuthority.issue_serial_number(issuer_entity)
    
        version = X509Builder.build_version()
        
        serial_number = X509Builder.build_serial_number(serial)
    
    
        # ---------------------------------------
        # Distinguished Names
        # ---------------------------------------
    
    
        issuer = X509Builder.build_name(issuer_entity['name'])
    
        subject = X509Builder.build_name(subject_entity['name'])
    
    
    
        # ---------------------------------------
        # Validity
        # ---------------------------------------
    
    
        issue_time = datetime.now(timezone.utc)
    
        expiry_time = issue_time + timedelta(days = validity_days)
        
        validity = X509Builder.build_validity(issue_time, expiry_time)
    
        
        # ---------------------------------------
        # Subject Public Key Info
        # ---------------------------------------

        spki_info = {
            "algorithm": subject_entity['signature']['algorithm'],
            "public_key": subject_entity['signature']['public_key']
        }
    
        spki = X509Builder.build_subject_public_key_info(
            subject_entity['signature']['algorithm'],
            subject_entity['signature']['public_key']
        )
    
        # ---------------------------------------
        # Extensions
        # ---------------------------------------
    
        encoded_extensions, extensions_data = X509Builder.build_default_extensions(issuer_entity, subject_entity)
    
        
        
    
        # ---------------------------------------
        # TBSCertificate
        # ---------------------------------------
    
        tbs = X509Builder.build_tbs_certificate(
            version = version,
            serial_number = serial_number,
            issuer=issuer,
            subject=subject,
            validity=validity,
            subject_public_key_info=spki,
            signature_algorithm=issuer_entity['signature']['algorithm'],
            extensions = encoded_extensions
        )

        print(tbs[:40].hex(" "))
    
        # ---------------------------------------
        # Sign Certificate
        # ---------------------------------------
    
        signature = CryptoManager.sign(issuer_entity, tbs)
            
        # ---------------------------------------
        # Build Certificate
        # ---------------------------------------
    
        certificate = X509Builder.build_certificate(
            tbs_certificate=tbs,
            signature_algorithm=issuer_entity['signature']['algorithm'],
            signature=signature
        )
    
        # ---------------------------------------
        # Store Certificate
        # ---------------------------------------

        issuer_entity["statistics"]["issued_certificates"] += 1
    
    
        certificate_record = {
    
            "serial_number": serial,

            "issuer_id": issuer_entity["entity_id"],
        
            "issuer": issuer_entity["common_name"],
        
            "subject": subject_entity["common_name"],
        
            "certificate": certificate,
        
            "tbs_certificate": tbs,
        
            "signature": signature,
        
            "status": "VALID",

            "revocation_date": None,

            "revocation_reason": None,
        
            "issued_at": issue_time,
        
            "expires_at": expiry_time,

            "subject_public_key_info" : spki_info,

            "extensions" : extensions_data,

            "self_signed": issuer_entity is subject_entity
        
        }
    
        subject_entity['certificate'] = certificate_record

        if issuer_entity is not subject_entity:
            subject_entity["parent_ca"] = issuer_entity
    
        issuer_entity['issued_certificates'].append(certificate_record)
    
        return certificate


    @staticmethod
    def get_certificate(
        issuer_entity: dict,
        serial_number: int
    ):
    
        for certificate in issuer_entity["issued_certificates"]:
    
            if certificate["serial_number"] == serial_number:
    
                return certificate
    
        return None



    @staticmethod
    def issue_crl(issuer_entity:dict, validity_days: int = 30) -> bytes:
        

        """
        Issue an X.509 Certificate Revocation List (CRL).
        
        Parameters
        ----------
        issuer_entity : dict
            Certificate Authority issuing the CRL.
        
        validity_days : int
            Number of days until the next CRL update.
        
        Returns
        -------
        bytes
            DER encoded Certificate Revocation List.
        """
        


        # ---------------------------------------
        # Validate cryptographic identity
        # ---------------------------------------

        

        if issuer_entity["signature"] is None:
            raise ValueError("Issuer has no Cryptographic Identity!!! ")
        
        

            

        # ---------------------------------------
        # Collect Revoked Certificates
        # ---------------------------------------


        revoked_records = CertificateAuthority.list_revoked_certificates(issuer_entity)



        # ---------------------------------------
        # Version
        # ---------------------------------------


        version = X509Builder.build_crl_version(2)



        # ---------------------------------------
        # Distinguished Name
        # ---------------------------------------


        issuer = X509Builder.build_name(issuer_entity["name"])



        # ---------------------------------------
        # Signature Algorithm
        # ---------------------------------------



        signature_algorithm = issuer_entity["signature"]["algorithm"]




        # ---------------------------------------
        # Update Times
        # ---------------------------------------



        this_update = datetime.now(timezone.utc)

        next_update = this_update + timedelta(days = validity_days)



        # ---------------------------------------
        # Revoked Certificates
        # ---------------------------------------



        revoked_certificates, revoked_certificates_data = X509Builder.build_revoked_certificates(revoked_records=revoked_records)



        # ---------------------------------------
        # Extensions
        # ---------------------------------------


        crl_extensions, crl_extensions_data = X509Builder.build_crl_extensions(issuer_entity)

        

        # ---------------------------------------
        # TBS CRL
        # ---------------------------------------



        tbs_crl = X509Builder.build_tbs_crl(
            
            version=version, 
            signature_algorithm=signature_algorithm, 
            issuer=issuer, 
            this_update=this_update, 
            next_update=next_update,
            revoked_certificates=revoked_certificates,
            extensions=crl_extensions
        
        )



        # ---------------------------------------
        # Sign Certificates
        # ---------------------------------------



        signature = CryptoManager.sign(issuer_entity, tbs_crl)



        # ---------------------------------------
        # Sign Certificates
        # ---------------------------------------



        crl = X509Builder.build_crl(tbs_crl, signature_algorithm, signature)


        


        # ---------------------------------------
        # Store it
        # ---------------------------------------

        



        crl_record = {"issuer": issuer_entity["common_name"],
        
            "this_update": this_update,
        
            "next_update": next_update,
        
            "algorithm": signature_algorithm,
        
            "signature": signature,
        
            "revoked_count": len(revoked_records),

            "revoked_certificates": revoked_certificates_data,

            "tbs_crl" : tbs_crl,

            "extensions": crl_extensions_data,
        
            "crl": crl}

        


        issuer_entity["crls"].append(crl_record)

        issuer_entity["next_crl_number"] += 1
        issuer_entity["statistics"]["generated_crls"] += 1

        return crl
        



    @staticmethod
    def revoke_certificate(entity: dict,  reason: RevocationReason):

        """
            Revoke a Certificate
        """

        certificate = entity["certificate"]

        if certificate is None:
            raise ValueError("Entity does not have a certificate!!!")

        issuer = entity["parent_ca"] or entity

        if certificate["status"] == "REVOKED":
            raise ValueError("Certificate already Revoked!!!")

        certificate["status"] = "REVOKED"

        certificate["revocation_date"] = datetime.now(timezone.utc)

        certificate["revocation_reason"] = reason

        issuer["statistics"]["revoked_certificates"] += 1

        


    @staticmethod
    def list_revoked_certificates(issuer:dict):

        """
        Return a list of all revoked certificates.
        
        Returns
        -------
        list
            List containing all revocation records.
        """


        revoked = []

        for certificate in issuer["issued_certificates"]:
            if certificate["status"] == "REVOKED":
                revoked.append(certificate)

        return revoked
        

    @staticmethod
    def is_revoked(entity: dict) -> bool:

        

        certificate = entity["certificate"]

        return certificate["status"] == "REVOKED"


    @staticmethod
    def get_latest_crl(issuer: dict):

        if not issuer["crls"]:
            return None

        return issuer["crls"][-1]


    @staticmethod
    def display_revoked_certificates(issuer:dict):
        """
        Display all revoked certificates.
        """
    
        revoked =  CertificateAuthority.list_revoked_certificates(issuer)
    
        print("\n" + "═" * 60)
        print("        " + issuer["common_name"])
        print("═" * 60)
    
        print(f"\nRevoked Certificates : {len(revoked)}")
    
        if not revoked:
            print("\nNo revoked certificates.")
            print("═" * 60)
            return
    
        for index, record in enumerate(revoked, start=1):
    
            print("\n" + "-" * 60)
    
            print(f"Subject          : {record['subject']}")
            print(f"Issuer           : {record['issuer']}")
            print(f"Issuer ID        : {record['issuer_id']}")
            print(f"Serial Number    : {record['serial_number']}")
            print(f"Revocation Date  : {record['revocation_date']}")
            print(f"Reason           : {record['revocation_reason'].value}")
    
        print("\n" + "═" * 60)

        


        
        


    @staticmethod
    def issue_serial_number(issuer_entity: dict) -> int:
        
        serial = issuer_entity["next_serial_number"]
        issuer_entity["next_serial_number"] += 1
    
        return serial

In [11]:
class X509CertificateValidator:

    @staticmethod
    def check_validity_period(certificate: dict) -> bool:

        

        issued_at = certificate["issued_at"]
        expires_at = certificate["expires_at"]

        now = datetime.now(timezone.utc)

        return issued_at <= now <= expires_at

        

    @staticmethod
    def check_basic_constraints(certificate: dict) -> bool:

        extensions = certificate["extensions"]

        basic_constraints = extensions.get("basic_constraints")

        if basic_constraints is None:
            return False
 

        ca = basic_constraints.get("ca")

        if not isinstance(ca,bool):
            return False

        
        if not ca:

            if "path_length" in basic_constraints:
                return False


            return True


        path_length = basic_constraints.get("path_length")

        if path_length is not None:

            if not isinstance(path_length, int):
                return False

            if path_length<0:
                return False

        return True

        


    @staticmethod
    def check_issuer_name(certificate, issuer_certificate):

        return certificate["issuer"] == issuer_certificate["subject"]

        
    @staticmethod
    def check_key_usage(certificate) -> bool:

        key_usage = certificate["extensions"]["key_usage"]

        basic_constraints = certificate["extensions"]["basic_constraints"]

        if "key_usage" not in certificate["extensions"]: 
            return False

        required_flags = [
            "digital_signature",
            "non_repudiation",
            "key_encipherment",
            "data_encipherment",
            "key_agreement",
            "key_cert_sign",
            "crl_sign",
            "encipher_only",
            "decipher_only"
        ]

        for flag in required_flags:

            if flag not in key_usage:
                return False

            if not isinstance(key_usage[flag], bool):
                return False

        if key_usage["key_cert_sign"] and not basic_constraints["ca"]:
            return False

        if key_usage["crl_sign"] and not basic_constraints["ca"]:
            return False


        if key_usage["encipher_only"] and not key_usage["key_agreement"]:
            return False

        if key_usage["decipher_only"] and not key_usage["key_agreement"]:
            return False

        return True



    @staticmethod
    def check_subject_key_identifier(certificate: dict) -> bool:
    
        extensions = certificate["extensions"]
    
        ski = extensions.get("subject_key_identifier")
        if ski is None:
            return False
    
        subject_public_key_info = certificate.get("subject_public_key_info")
        if subject_public_key_info is None:
            return False
    
        public_key = subject_public_key_info.get("public_key")
        if public_key is None:
            return False
    
        expected_ski = hashlib.sha1(public_key).digest()
    
        return expected_ski == ski


    @staticmethod
    def check_authority_key_identifier(certificate: dict, issuer_certificate: dict)->bool:

        extensions = certificate["extensions"]

        issuer_extensions = issuer_certificate["extensions"]

        if "authority_key_identifier" not in extensions:
            return False
        
        if "subject_key_identifier" not in issuer_extensions:
            return False


        aki = extensions["authority_key_identifier"]

        issuer_ski = issuer_extensions["subject_key_identifier"]

        return aki == issuer_ski


    @staticmethod
    def check_signature(certificate: dict, issuer: dict) -> bool:

        if "tbs_certificate" not in certificate:
            return False

        if "signature" not in certificate:
            return False

        tbs = certificate["tbs_certificate"]
        signature = certificate["signature"]


        try:
            
            return CryptoManager.verify(data=tbs, entity=issuer, signature=signature)

        except ValueError:

            return False


    @staticmethod
    def validate_certificate(certificate: dict, issuer: dict) -> bool:

        issuer_certificate = issuer["certificate"]

        return(

            X509CertificateValidator.check_validity_period(certificate)

            and 

            X509CertificateValidator.check_basic_constraints(certificate)

            and

            X509CertificateValidator.check_key_usage(certificate)

            and 

            X509CertificateValidator.check_subject_key_identifier(certificate)

            and 

            X509CertificateValidator.check_authority_key_identifier(certificate, issuer_certificate)

            and 

            X509CertificateValidator.check_signature(certificate, issuer)
        )

        


        
        

In [12]:
# ==========================================================
# SECTION 10 : TRUST STORE
# ==========================================================


class TrustStore:
    """
    QuantumSoft PKI Trust Store

    Stores and manages trusted Root Certificate Authorities.

    Responsibilities
    ----------------
    - Register trusted Root CAs
    - Remove trusted Root CAs
    - Lookup trusted Root CAs
    - Check trust status
    - Display Trust Store information

    Future Responsibilities
    -----------------------
    - Certificate Chain Validation
    - Signature Verification
    - Revocation Checking
    - Certificate Policy Validation
    """

    def __init__(self):


        #UUID -> Root Entity
        self.trusted_roots = {}




    def add_root(self, root_entity):

        """
        Register a Trusted Root CA
        """


        entity_id = root_entity["entity_id"]
        
        if entity_id in self.trusted_roots:

            raise ValueError(f"Root CA '{root_entity['common_name']}' is already trusted!")

        self.trusted_roots[entity_id] = root_entity
        

        print(f"Trusted Root Added: {root_entity['common_name']}")




    def remove_root(self, entity_id):

        if entity_id not in self.trusted_roots:

            raise ValueError("Root CA not found!!!!")


        root_name = self.trusted_roots[entity_id]["common_name"]


        del self.trusted_roots[entity_id]

        print(f"Removed Trusted Root: {root_name}")
        return root_name




    def get_root(self, entity_id):

        return self.trusted_roots.get(entity_id)




    def is_root_trusted(self, entity_id):

        return entity_id in self.trusted_roots




    def list_roots(self):

        return list(self.trusted_roots.values())



    def display(self):

        print("\n" + "═" * 60)
        print("           QuantumSoft Trust Store")
        print("═" * 60)

        print(f"\nTrusted Roots : {len(self.trusted_roots)}")

        if not self.trusted_roots:
            print("\nNo trusted roots registered.")
            return

        for index, root in enumerate(self.trusted_roots.values(), start=1):

            print("\n" + "-" * 60)

            print(f"Root #{index}")
            print(f"Common Name  : {root['common_name']}")
            print(f"Organization : {root['organization']}")
            print(f"Entity ID    : {root['entity_id']}")
            print(f"Algorithm    : {root['signature']['algorithm']}")

        print("\n" + "═" * 60)



    def build_chain(self, entity):

        """
        Build the certification path from an entity
        to the trusted Root CA.
        """

        chain = []

        current = entity

        visited = set()

        while current is not None:

            if current["entity_id"] in visited:

                raise ValueError("Cycle Detected!!!")

            visited.add(current["entity_id"])

            chain.append(current)


            current = current.get("parent_ca")



        return chain



In [13]:
from dataclasses import dataclass

@dataclass
class ValidationResult:

    valid: bool

    stage: str

    message: str

    certificate: dict = None

    issuer: dict = None

In [14]:
class PathValidator:

    @staticmethod
    def validate_certificate_chain(entity,trust_store):

        
        chain = trust_store.build_chain(entity)

        if not chain:
            return ValidationResult(valid = False, stage = "Chain Building", message = "Unable to build Certificate Path!!")

        for i in range(len(chain) - 1):

            subject = chain[i]
            issuer = chain[i+1]

            if not X509CertificateValidator.validate_certificate(subject["certificate"], issuer):
                
                return ValidationResult(valid=False, stage = "Certificate Validation", message=f"Validation failed for {subject['common_name']}!!!", certificate=subject,issuer=issuer)


            if CertificateAuthority.is_revoked(subject):

                return ValidationResult(valid = False, stage="Revocation Check", message=f"{subject['common_name']} certificate has been revoked!!!", certificate=subject, issuer=issuer)


        #Verify self-signed root

        root = chain[-1]
        if not X509CertificateValidator.validate_certificate(root["certificate"], root):
            
            return ValidationResult(valid=False, stage = "Root Validation", message="Root Certificate is invalid!!!", certificate=root)

        if not trust_store.is_root_trusted(root["entity_id"]):
            
            return ValidationResult(valid=False, stage="Trust Anchor", message="Root CA is not trusted!!!")


        
        return ValidationResult(valid=True, stage = "Completed", message="Certificate Chain Successfully Validated!!!", certificate=entity)

In [15]:
class PKIFramework:

    def __init__(self):

        self

        self.entities = {}

        self.statistics = {
            "successful_validations": 0,
            "failed_validations": 0
        }

        self.trust_store = TrustStore()

        self.path_validator = PathValidator()



    def add_entity(self, entity):

        entity_id = entity["entity_id"]

        if entity_id in self.entities:
            raise ValueError("Entity Already Exists!!!")
            
        self.entities[entity_id] = entity

        


    def remove_entity(self, entity_id):

        if entity_id not in self.entities:

            raise ValueError("Unknown Entity!!!")


        del self.entities[entity_id]

        


    def get_entity(self, entity_id):

        return self.entities.get(entity_id)



    def list_entities(self):

        return list(self.entities.values())


    def register_root(self, root):

        self.add_entity(root)

        self.trust_store.add_root(root)

        


    def unregister_root(self, entity_id):

        self.trust_store.remove_root(entity_id)


    def get_entity(self, entity_id):

        return self.entities.get(entity_id)



    def display(self):
    
        print("="*60)
        print("               PKI FRAMEWORK")
        print("="*60)
    
        print(f"Entities        : {len(self.entities)}")
    
        print(f"Trusted Roots   : {len(self.trust_store.trusted_roots)}")
    
        print("="*60)


    def validate(self, entity):

        result = self.path_validator.validate_certificate_chain(entity, self.trust_store)

        if result.valid:
            self.statistics['successful_validations'] += 1

        else:
            self.statistics['failed_validations'] += 1


        return result



    

In [16]:
@dataclass
class SignedMessage:

    sender_id: str

    receiver_id: str

    payload: bytes

    signature: bytes

    certificate: dict

    timestamp: datetime


@dataclass
class MessageVerificationResult:

    valid: bool

    stage: str

    message: str

    sender: dict | None

    receiver: dict | None

    signed_message: SignedMessage | None


@dataclass
class EncryptedMessage:

    sender_id: str

    receiver_id: str

    encrypted_payload: bytes

    encrypted_session_key: bytes

    nonce: bytes

    algorithm: str

    timestamp: datetime

    

In [17]:
class CommunicationManager:

    def __init__(self, pki_framework):

        self.pki = pki_framework



    def create_signed_message(self, sender: dict, receiver: dict, payload: str | bytes) -> SignedMessage:

        if isinstance(payload, str):
            payload = payload.encode("utf-8")


        signature = CryptoManager.sign(entity=sender, data=payload)

        return SignedMessage(
            sender_id=sender["entity_id"],
            receiver_id=receiver["entity_id"],
            payload=payload,
            signature=signature,
            certificate=sender["certificate"],
            timestamp=datetime.now(timezone.utc)
        )



    def verify_signed_message(self, receiver: dict, signed_message:SignedMessage) -> MessageVerificationResult:

        sender = self.pki.get_entity(signed_message.sender_id)

        if sender is None:

            return MessageVerificationResult(
                valid=False,
                stage="Sender Lookup",
                message="Sender Not Registered",
                sender=None,
                receiver=receiver,
                signed_message=signed_message
            )


        certificate_result = self.pki.validate(sender)

        if not certificate_result.valid:

            return MessageVerificationResult(
                valid=False,
                stage="Certificate Validation",
                message="Sender Certificate is invalid",
                sender=sender,
                receiver=receiver,
                signed_message=signed_message
            )


        signature_valid = CryptoManager.verify(entity=sender, data=signed_message.payload, signature=signed_message.signature)

        if not signature_valid:

            return MessageVerificationResult(
                valid=False, 
                stage="Signature Verification",
                message="Digital signature verification failed",
                sender=sender,
                receiver=receiver,
                signed_message=signed_message
            )


        return MessageVerificationResult(
            valid=True,
            stage="Completed",
            message="Message Successfully Authenticated",
            sender=sender,
            receiver=receiver,
            signed_message=signed_message
        )


    

        

In [18]:
root_ca = EntityFactory.create_root_ca(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="QuantumSoft Root CA"
)

employee_ca = EntityFactory.create_intermediate_ca(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="QuantumSoft Employee Intermediate CA"
)

server_ca = EntityFactory.create_intermediate_ca(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="QuantumSoft Server Intermediate CA"
)

alice = EntityFactory.create_employee(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="Alice",
    employee_id="EMP001"
)

bob = EntityFactory.create_employee(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="Bob",
    employee_id="EMP002"
)

web_server = EntityFactory.create_server(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="Web Server",
    dns_name="www.quantumsoft.com"
)

mail_server = EntityFactory.create_server(
    country="IN",
    state="Delhi",
    locality="New Delhi",
    organization="QuantumSoft Pvt. Ltd.",
    common_name="Mail Server",
    dns_name="mail.quantumsoft.com"
)

In [19]:
EntityFactory.display(root_ca)
EntityFactory.display(employee_ca)
EntityFactory.display(server_ca)
EntityFactory.display(alice)
EntityFactory.display(bob)
EntityFactory.display(web_server)
EntityFactory.display(mail_server)

ROOT_CA INFORMATION
Entity ID      : c1b105a5-dc1a-48e9-9333-2b045731d8d4
Common Name    : QuantumSoft Root CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391141+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Root CA
organization   : QuantumSoft Pvt. Ltd.
country        : IN
state          : Delhi
locality       : New Delhi


Signature      : Not Initialized
Certificate    : Not Issued
Parent CA      : None

CA INFORMATION
------------------
Issued Certs   : 0
CRLs           : 0
Next Serial    : 1

Statistics
----------
issued_certificates      : 0
revoked_certificates     : 0
generated_crls           : 0
INTERMEDIATE_CA INFORMATION
Entity ID      : aa0c4eb1-56e9-4a5f-928c-86124d9affb9
Common Name    : QuantumSoft Employee Intermediate CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391383+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Employee Intermediate CA
organizati

In [20]:
ALGORITHM = "ML_DSA_65"      # or Dilithium3 depending on your implementation

CryptoManager.initialize_signature(root_ca, ALGORITHM)

CryptoManager.initialize_signature(employee_ca, ALGORITHM)

CryptoManager.initialize_signature(server_ca, ALGORITHM)

CryptoManager.initialize_signature(alice, ALGORITHM)

CryptoManager.initialize_signature(bob, ALGORITHM)

CryptoManager.initialize_signature(web_server, ALGORITHM)

CryptoManager.initialize_signature(mail_server, ALGORITHM)

print()

In [21]:
EntityFactory.display(root_ca)

EntityFactory.display(employee_ca)

EntityFactory.display(server_ca)

EntityFactory.display(alice)

EntityFactory.display(bob)

EntityFactory.display(web_server)

EntityFactory.display(mail_server)

ROOT_CA INFORMATION
Entity ID      : c1b105a5-dc1a-48e9-9333-2b045731d8d4
Common Name    : QuantumSoft Root CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391141+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Root CA
organization   : QuantumSoft Pvt. Ltd.
country        : IN
state          : Delhi
locality       : New Delhi


Signature      : Initialized
Certificate    : Not Issued
Parent CA      : None

CA INFORMATION
------------------
Issued Certs   : 0
CRLs           : 0
Next Serial    : 1

Statistics
----------
issued_certificates      : 0
revoked_certificates     : 0
generated_crls           : 0
INTERMEDIATE_CA INFORMATION
Entity ID      : aa0c4eb1-56e9-4a5f-928c-86124d9affb9
Common Name    : QuantumSoft Employee Intermediate CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391383+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Employee Intermediate CA
organization  

In [22]:
# Root self-signed
CertificateAuthority.issue_certificate(root_ca, root_ca)

# Root issues Intermediate CAs
CertificateAuthority.issue_certificate(root_ca, employee_ca)
CertificateAuthority.issue_certificate(root_ca, server_ca)

# Employee CA issues employee certificates
CertificateAuthority.issue_certificate(employee_ca, alice)
CertificateAuthority.issue_certificate(employee_ca, bob)

# Server CA issues server certificates
CertificateAuthority.issue_certificate(server_ca, web_server)
CertificateAuthority.issue_certificate(server_ca, mail_server)

print()

30 82 09 36 a0 03 02 01 02 02 01 01 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 6f 31 0b 30 09 06 03 55 04 06 13 02 49 4e
30 82 09 48 a0 03 02 01 02 02 01 02 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 6f 31 0b 30 09 06 03 55 04 06 13 02 49 4e
30 82 09 45 a0 03 02 01 02 02 01 03 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 6f 31 0b 30 09 06 03 55 04 06 13 02 49 4e
30 82 09 37 a0 03 02 01 02 02 01 01 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 81 80 31 0b 30 09 06 03 55 04 06 13 02 49
30 82 09 35 a0 03 02 01 02 02 01 02 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 81 80 31 0b 30 09 06 03 55 04 06 13 02 49
30 82 09 39 a0 03 02 01 02 02 01 01 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 7e 31 0b 30 09 06 03 55 04 06 13 02 49 4e
30 82 09 3a a0 03 02 01 02 02 01 02 30 0b 06 09 60 86 48 01 65 03 04 03 12 30 7e 31 0b 30 09 06 03 55 04 06 13 02 49 4e



In [23]:
EntityFactory.display(root_ca)

EntityFactory.display(employee_ca)

EntityFactory.display(server_ca)

EntityFactory.display(alice)

EntityFactory.display(bob)

EntityFactory.display(web_server)

EntityFactory.display(mail_server)

ROOT_CA INFORMATION
Entity ID      : c1b105a5-dc1a-48e9-9333-2b045731d8d4
Common Name    : QuantumSoft Root CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391141+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Root CA
organization   : QuantumSoft Pvt. Ltd.
country        : IN
state          : Delhi
locality       : New Delhi


Signature      : Initialized
Certificate    : Issued
Parent CA      : None

CA INFORMATION
------------------
Issued Certs   : 3
CRLs           : 0
Next Serial    : 4

Statistics
----------
issued_certificates      : 3
revoked_certificates     : 0
generated_crls           : 0
INTERMEDIATE_CA INFORMATION
Entity ID      : aa0c4eb1-56e9-4a5f-928c-86124d9affb9
Common Name    : QuantumSoft Employee Intermediate CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391383+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Employee Intermediate CA
organization   : Q

In [24]:
certificate = CertificateAuthority.get_certificate(
    root_ca,
    99
)

print(certificate)

None


In [25]:
certificate = CertificateAuthority.get_certificate(root_ca, 2)

print(f"Serial Number : {certificate['serial_number']}")
print(f"Issuer        : {certificate['issuer']}")
print(f"Subject       : {certificate['subject']}")
print(f"Status        : {certificate['status']}")
print(f"Self Signed   : {certificate['self_signed']}")
print(f"Issued At     : {certificate['issued_at']}")
print(f"Expires At    : {certificate['expires_at']}")

Serial Number : 2
Issuer        : QuantumSoft Root CA
Subject       : QuantumSoft Employee Intermediate CA
Status        : VALID
Self Signed   : False
Issued At     : 2026-08-04 19:17:45.119010+00:00
Expires At    : 2027-08-04 19:17:45.119010+00:00


In [26]:
print(f"Certificate Size     : {len(certificate['certificate'])} bytes")
print(f"TBS Size             : {len(certificate['tbs_certificate'])} bytes")
print(f"Signature Size       : {len(certificate['signature'])} bytes")

Certificate Size     : 5711 bytes
TBS Size             : 2380 bytes
Signature Size       : 3309 bytes


In [27]:
certificate = alice["certificate"]

print(f"Serial Number : {certificate['serial_number']}")
print(f"Status        : {certificate['status']}")

Serial Number : 1
Status        : VALID


In [28]:
print(f"Status : {alice['certificate']['status']}")

Status : VALID


In [29]:
pki = PKIFramework()

In [30]:
pki.register_root(root_ca)

Trusted Root Added: QuantumSoft Root CA


In [31]:
pki.add_entity(employee_ca)

pki.add_entity(server_ca)

pki.add_entity(alice)

pki.add_entity(bob)

pki.add_entity(web_server)

pki.add_entity(mail_server)

In [32]:
pki.validate(alice)

pki.validate(bob)

pki.validate(web_server)

pki.validate(mail_server)

print()

In [33]:
comm = CommunicationManager(pki)


msg = comm.create_signed_message(sender=alice, receiver=web_server, payload="Hey! Alice this side!")

In [34]:
pprint(msg)

SignedMessage(sender_id='9542cae4-d4a2-4b1a-add2-9dbf97327a74',
              receiver_id='7961532c-6989-4bbf-b480-d2ecaccd169f',
              payload=b'Hey! Alice this side!',
              signature=b'\xe6\x0b\xfa\xde\xa2:\xb5d\xf2\x92\t\xdf\x81\xf9\n4'
                        b'\x8a\xaf\x9e\x86\xbaT\x11:\xb6\xf0\x9c\x9cBx\xc2\\'
                        b'\xba\xb79\x98h\x1fX=_o!\x11\xce\xe25:\x0fww\xca'
                        b'\x16\xbc\x03Z\x89h\x05\xd0q\x82\xca2\x8c:\xbfq'
                        b'\xaa/J\xd5!\xd8\xa0\xc6\xcbR\x1fc\xd8\x94^\x9e'
                        b'Y_\xdb\xf7\xa9\xcc)\xda\x1b\x0cs\x08u\xeb\xb00'
                        b'~\x9e]\xb6\x03\xa6\xc7}Boi\xcdG\xc1\xb2_`\x86l\r'
                        b'\x98\x7f\xff\x9b\xe4\x1e\x92\x7f\xfc\xda1a\xc0VS\xe4'
                        b'8\x11\xf5\xff\xd9\x86\xd99\xc4\xa6J\xdfC\x89\x91|'
                        b"\x83u\xccth\x19J\x14\x1e\x88\x8e\xb4$\xdfrz'\x0e\xaf<"
                        b'\x06#\xcc\x9bRez\xf2[\x11x\x

In [34]:
result = comm.verify_signed_message(receiver=web_server, signed_message=msg)


print(result.valid)
print(result.stage)
print(result.message)

True
Completed
Message Successfully Authenticated


In [35]:
CertificateAuthority.revoke_certificate(
    entity=alice,
    reason=RevocationReason.KEY_COMPROMISE
)

In [36]:
pki.validate(alice)

ValidationResult(valid=False, stage='Revocation Check', message='Alice certificate has been revoked!!!', certificate={'entity_id': '6258760e-f2db-4c95-aeae-d30b8f0dce76', 'type': 'EMPLOYEE', 'organization': 'QuantumSoft Pvt. Ltd.', 'common_name': 'Alice', 'name': {'common_name': 'Alice', 'organization': 'QuantumSoft Pvt. Ltd.', 'country': 'IN', 'state': 'Delhi', 'locality': 'New Delhi'}, 'signature': {'algorithm': 'ML_DSA_65', 'family': 'ML_DSA', 'hash': 'SHA3-384', 'oqs_name': 'ML-DSA-65', 'oid': '2.16.840.1.101.3.4.3.18', 'public_key': b'\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<\xd4}<N\xfcK\x0c?Tc\x0e\x041\xd3\xc5X7\xd8|a\xcb\xc8\x85A\xfaC\xc2\xffs\xe2\x13\xd028\x9a\xd9\xeds\'\xa7+\xd8\xb3\x0b\x8e\x998\x18\x8b\xb71L\xfeGL\xce#{\xd4\x08\xe9\xed[~\x8a&T\x06B\xee\x11\xe3\x0eyhX\x02\xca\xd0\x8b\xca\xce\xa7@\xc0\x05\xc3G\x80\xbf\xd9H<\x98Q\xa1\xa8\x84

In [37]:
alice["certificate"]["signature"] = b"tampered"

In [38]:
pki.validate(alice)

ValidationResult(valid=False, stage='Certificate Validation', message='Validation failed for Alice!!!', certificate={'entity_id': '6258760e-f2db-4c95-aeae-d30b8f0dce76', 'type': 'EMPLOYEE', 'organization': 'QuantumSoft Pvt. Ltd.', 'common_name': 'Alice', 'name': {'common_name': 'Alice', 'organization': 'QuantumSoft Pvt. Ltd.', 'country': 'IN', 'state': 'Delhi', 'locality': 'New Delhi'}, 'signature': {'algorithm': 'ML_DSA_65', 'family': 'ML_DSA', 'hash': 'SHA3-384', 'oqs_name': 'ML-DSA-65', 'oid': '2.16.840.1.101.3.4.3.18', 'public_key': b'\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<\xd4}<N\xfcK\x0c?Tc\x0e\x041\xd3\xc5X7\xd8|a\xcb\xc8\x85A\xfaC\xc2\xffs\xe2\x13\xd028\x9a\xd9\xeds\'\xa7+\xd8\xb3\x0b\x8e\x998\x18\x8b\xb71L\xfeGL\xce#{\xd4\x08\xe9\xed[~\x8a&T\x06B\xee\x11\xe3\x0eyhX\x02\xca\xd0\x8b\xca\xce\xa7@\xc0\x05\xc3G\x80\xbf\xd9H<\x98Q\xa1\xa8\x84\

In [39]:
alice["certificate"]["expires_at"] = datetime.now(timezone.utc) - timedelta(days=1)

In [40]:
pki.validate(alice)

ValidationResult(valid=False, stage='Certificate Validation', message='Validation failed for Alice!!!', certificate={'entity_id': '6258760e-f2db-4c95-aeae-d30b8f0dce76', 'type': 'EMPLOYEE', 'organization': 'QuantumSoft Pvt. Ltd.', 'common_name': 'Alice', 'name': {'common_name': 'Alice', 'organization': 'QuantumSoft Pvt. Ltd.', 'country': 'IN', 'state': 'Delhi', 'locality': 'New Delhi'}, 'signature': {'algorithm': 'ML_DSA_65', 'family': 'ML_DSA', 'hash': 'SHA3-384', 'oqs_name': 'ML-DSA-65', 'oid': '2.16.840.1.101.3.4.3.18', 'public_key': b'\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<\xd4}<N\xfcK\x0c?Tc\x0e\x041\xd3\xc5X7\xd8|a\xcb\xc8\x85A\xfaC\xc2\xffs\xe2\x13\xd028\x9a\xd9\xeds\'\xa7+\xd8\xb3\x0b\x8e\x998\x18\x8b\xb71L\xfeGL\xce#{\xd4\x08\xe9\xed[~\x8a&T\x06B\xee\x11\xe3\x0eyhX\x02\xca\xd0\x8b\xca\xce\xa7@\xc0\x05\xc3G\x80\xbf\xd9H<\x98Q\xa1\xa8\x84\

In [41]:
alice["certificate"]["extensions"]["subject_key_identifier"] = b"\x00" * 20

In [42]:
pki.validate(alice)

ValidationResult(valid=False, stage='Certificate Validation', message='Validation failed for Alice!!!', certificate={'entity_id': '6258760e-f2db-4c95-aeae-d30b8f0dce76', 'type': 'EMPLOYEE', 'organization': 'QuantumSoft Pvt. Ltd.', 'common_name': 'Alice', 'name': {'common_name': 'Alice', 'organization': 'QuantumSoft Pvt. Ltd.', 'country': 'IN', 'state': 'Delhi', 'locality': 'New Delhi'}, 'signature': {'algorithm': 'ML_DSA_65', 'family': 'ML_DSA', 'hash': 'SHA3-384', 'oqs_name': 'ML-DSA-65', 'oid': '2.16.840.1.101.3.4.3.18', 'public_key': b'\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<\xd4}<N\xfcK\x0c?Tc\x0e\x041\xd3\xc5X7\xd8|a\xcb\xc8\x85A\xfaC\xc2\xffs\xe2\x13\xd028\x9a\xd9\xeds\'\xa7+\xd8\xb3\x0b\x8e\x998\x18\x8b\xb71L\xfeGL\xce#{\xd4\x08\xe9\xed[~\x8a&T\x06B\xee\x11\xe3\x0eyhX\x02\xca\xd0\x8b\xca\xce\xa7@\xc0\x05\xc3G\x80\xbf\xd9H<\x98Q\xa1\xa8\x84\

In [43]:
certificate = alice["certificate"]

print(f"Status            : {certificate['status']}")
print(f"Revocation Date   : {certificate['revocation_date']}")
print(f"Revocation Reason : {certificate['revocation_reason']}")

Status            : REVOKED
Revocation Date   : 2026-08-04 19:18:15.721897+00:00
Revocation Reason : 1


In [44]:
EntityFactory.display(employee_ca)

INTERMEDIATE_CA INFORMATION
Entity ID      : aa0c4eb1-56e9-4a5f-928c-86124d9affb9
Common Name    : QuantumSoft Employee Intermediate CA
Organization   : QuantumSoft Pvt. Ltd.
Created At     : 2026-08-04 19:17:40.391383+00:00

Distinguished Name
------------------
common_name    : QuantumSoft Employee Intermediate CA
organization   : QuantumSoft Pvt. Ltd.
country        : IN
state          : Delhi
locality       : New Delhi


Signature      : Initialized
Certificate    : Issued
Parent CA      : QuantumSoft Root CA

CA INFORMATION
------------------
Issued Certs   : 2
CRLs           : 0
Next Serial    : 3

Statistics
----------
issued_certificates      : 2
revoked_certificates     : 1
generated_crls           : 0


In [45]:
print(
    CertificateAuthority.is_revoked(alice)
)

True


In [46]:
print(
    CertificateAuthority.is_revoked(bob)
)

False


In [47]:
print(
    CertificateAuthority.is_revoked(web_server)
)

False


In [48]:
pki.unregister_root(root_ca["entity_id"])

Removed Trusted Root: QuantumSoft Root CA


In [49]:
pki.validate(alice)

ValidationResult(valid=False, stage='Certificate Validation', message='Validation failed for Alice!!!', certificate={'entity_id': '6258760e-f2db-4c95-aeae-d30b8f0dce76', 'type': 'EMPLOYEE', 'organization': 'QuantumSoft Pvt. Ltd.', 'common_name': 'Alice', 'name': {'common_name': 'Alice', 'organization': 'QuantumSoft Pvt. Ltd.', 'country': 'IN', 'state': 'Delhi', 'locality': 'New Delhi'}, 'signature': {'algorithm': 'ML_DSA_65', 'family': 'ML_DSA', 'hash': 'SHA3-384', 'oqs_name': 'ML-DSA-65', 'oid': '2.16.840.1.101.3.4.3.18', 'public_key': b'\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<\xd4}<N\xfcK\x0c?Tc\x0e\x041\xd3\xc5X7\xd8|a\xcb\xc8\x85A\xfaC\xc2\xffs\xe2\x13\xd028\x9a\xd9\xeds\'\xa7+\xd8\xb3\x0b\x8e\x998\x18\x8b\xb71L\xfeGL\xce#{\xd4\x08\xe9\xed[~\x8a&T\x06B\xee\x11\xe3\x0eyhX\x02\xca\xd0\x8b\xca\xce\xa7@\xc0\x05\xc3G\x80\xbf\xd9H<\x98Q\xa1\xa8\x84\

In [50]:
print(CertificateAuthority.list_revoked_certificates(server_ca))

[]


In [51]:
crl = CertificateAuthority.issue_crl(server_ca)

print(f"CRL Length : {len(crl)} bytes")
print(f"Generated CRLs : {server_ca['statistics']['generated_crls']}")
print(f"Next CRL Number : {server_ca['next_crl_number']}")

CRL Length : 3559 bytes
Generated CRLs : 1
Next CRL Number : 2


In [52]:
server_ca["crls"][-1]["crl"]

b'0\x82\r\xe30\x81\xe1\x02\x01\x010\x0b\x06\t`\x86H\x01e\x03\x04\x03\x120~1\x0b0\t\x06\x03U\x04\x06\x13\x02IN1\x0e0\x0c\x06\x03U\x04\x08\x0c\x05Delhi1\x120\x10\x06\x03U\x04\x07\x0c\tNew Delhi1\x1e0\x1c\x06\x03U\x04\n\x0c\x15QuantumSoft Pvt. Ltd.1+0)\x06\x03U\x04\x03\x0c"QuantumSoft Server Intermediate CA\x17\r260804191835Z\x17\r260903191835Z0\x00\xa0/0-0\x1f\x06\x03U\x1d#\x04\x180\x16\x80\x14\xd0\xa4\xe9H\xaa\\\xc9\x95\xc6\xf9\x1a\x11H\xcd\xd1\x0f\xa0A\x81-0\n\x06\x03U\x1d\x14\x04\x03\x02\x01\x010\x0b\x06\t`\x86H\x01e\x03\x04\x03\x12\x03\x82\x0c\xee\x00\xd6!\x87S\xb2[\x84\x079\x1e\x96a\x9f\xdc|\x82!\x1e\x1e\x0f*t\x91\xc5B\x86\xdc\\:\xf5}\xb3u)}\xf5\xae\xadq\xb3sC\x1e@\x10\x8d\xdcK\xd1\xcbq3\x18\xb5\x98v\xd7\x84DJ\xce?\xaa\x84\x84\xd1\xeb\xcfT\x9d2\xba~(TH\xecT`l\x0e\xdd\x9d\xcbk\xad\xd0se\xcc\xbf\x83.\x99~\xd5+>8lQb\xed\xa6X\xa7V\xd1\\\xce|\x16H\xb6\xa9\xf2\x9dW\xaa\xaf\xd1\x07\x9aG!\xb7\xc2\x96\xf7\x87\x85\xe2;udrGh\xd0\xcd=\xa1\x93\xbd\x93\xc9\xd7>H\xafJ>\x87Q\xb2\xc2\x90\x9c,\xfd}ce

In [53]:
print(CertificateAuthority.list_revoked_certificates(employee_ca))

[{'serial_number': 1, 'issuer_id': 'aa0c4eb1-56e9-4a5f-928c-86124d9affb9', 'issuer': 'QuantumSoft Employee Intermediate CA', 'subject': 'Alice', 'certificate': b'0\x82\x16:0\x82\t7\xa0\x03\x02\x01\x02\x02\x01\x010\x0b\x06\t`\x86H\x01e\x03\x04\x03\x120\x81\x801\x0b0\t\x06\x03U\x04\x06\x13\x02IN1\x0e0\x0c\x06\x03U\x04\x08\x0c\x05Delhi1\x120\x10\x06\x03U\x04\x07\x0c\tNew Delhi1\x1e0\x1c\x06\x03U\x04\n\x0c\x15QuantumSoft Pvt. Ltd.1-0+\x06\x03U\x04\x03\x0c$QuantumSoft Employee Intermediate CA0\x1e\x17\r260804191745Z\x17\r270804191745Z0a1\x0b0\t\x06\x03U\x04\x06\x13\x02IN1\x0e0\x0c\x06\x03U\x04\x08\x0c\x05Delhi1\x120\x10\x06\x03U\x04\x07\x0c\tNew Delhi1\x1e0\x1c\x06\x03U\x04\n\x0c\x15QuantumSoft Pvt. Ltd.1\x0e0\x0c\x06\x03U\x04\x03\x0c\x05Alice0\x82\x07\xb20\x0b\x06\t`\x86H\x01e\x03\x04\x03\x12\x03\x82\x07\xa1\x00\x8dl\x9bk\x84\xa7\xcc/\xb6O\xdb\x9d\x9cB\xd3z\xf6\xf4\xac\x84\xd1\x14{\xe9\x8c\xe0N\x13~FJx\xc9\xc2\xbc\rea\xff\xd8\x80\x9c\x08\xf7CnF\xe9z\xa4i\n\xa6\xb9G\xcd\xe2\xdfhk G\x86\xd0<

In [54]:
crl = CertificateAuthority.issue_crl(employee_ca)

print(f"CRL Length : {len(crl)} bytes")
print(f"Generated CRLs : {employee_ca['statistics']['generated_crls']}")
print(f"Next CRL Number : {employee_ca['next_crl_number']}")

CRL Length : 3597 bytes
Generated CRLs : 1
Next CRL Number : 2


In [55]:
latest = employee_ca["crls"][-1]

print("Revoked Count:", latest["revoked_count"])
print("CRL Extensions:")
print(latest["extensions"])

print("\nRevoked Certificates:")
from pprint import pprint
pprint(latest["revoked_certificates"])

Revoked Count: 1
CRL Extensions:
{'authority_key_identifier': b'V\x0f\xda\xc0\\L=\xfe\x1e\x98\x1f\x01pF\x89U\xa4\xbe$\x9b', 'crl_number': 1}

Revoked Certificates:
[{'extensions': {'reason': 'KEY_COMPROMISE'},
  'revocation_date': datetime.datetime(2026, 8, 4, 19, 18, 15, 721897, tzinfo=datetime.timezone.utc),
  'serial_number': 1}]


In [56]:
result = CryptoManager.verify(
    employee_ca,
    employee_ca["crls"][-1]["tbs_crl"],
    employee_ca["crls"][-1]["signature"]
)

print(result)

True
